In [1]:
import os, random
import pandas as pd

from torch.utils.data import Subset
import numpy as np

from PIL import Image
Image.MAX_IMAGE_PIXELS = None
from PIL import ImageFile, UnidentifiedImageError
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms
import torchvision.models as models

from torchvision.models import resnet50, ResNet50_Weights

import numpy as np
from sklearn.metrics import f1_score, recall_score, confusion_matrix, classification_report

INPUT_DIR = "/kaggle/input/datasets/shashiprabhapanwar/wikiart"
IMAGES_ROOT = os.path.join(INPUT_DIR, "wikiart_clean", "wikiart")
TRAIN_CSV = os.path.join(INPUT_DIR, "style_train_clean.csv")
VAL_CSV   = os.path.join(INPUT_DIR, "style_val_lite_clean.csv")

In [2]:
def read_wikiart_csv(path):
    df = pd.read_csv(path)

    # headerless got misread as header (common)
    if len(df.columns) == 2:
        c0, c1 = df.columns[0], df.columns[1]
        if ("/" in str(c0)) and str(c1).strip().isdigit():
            df = pd.read_csv(path, header=None, names=["relpath", "label"])
            df["label"] = df["label"].astype(int)
            return df

    # normal header (strip spaces)
    df.columns = [c.strip() for c in df.columns]

    # guess columns
    path_col = df.columns[0]
    label_col = df.columns[1]
    for c in df.columns:
        lc = c.lower()
        if lc in ["pictures", "picture", "path", "file", "filename", "image"]:
            path_col = c
        if lc in ["class", "label", "target"]:
            label_col = c

    df = df[[path_col, label_col]].rename(columns={path_col:"relpath", label_col:"label"})
    df["label"] = df["label"].astype(int)
    return df


def filter_bad_images(df, images_root):
    good_rows = []
    bad = 0

    for relpath, label in zip(df["relpath"].astype(str).values, df["label"].values):
        fullpath = os.path.join(images_root, relpath)
        try:
            with Image.open(fullpath) as im:
                im.verify()  # verifies file integrity without decoding full image
            good_rows.append((relpath, int(label)))
        except (OSError, UnidentifiedImageError):
            bad += 1

    out = pd.DataFrame(good_rows, columns=["relpath", "label"])
    print(f"Filtered bad images: {bad} removed, {len(out)} remaining")
    return out

def evaluate_metrics(model, loader, device, use_amp=True, num_classes=None):
    model.eval()

    all_targets = []
    all_preds = []
    top5_correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            with torch.amp.autocast(device_type="cuda", enabled=use_amp):
                logits = model(x)

            pred = logits.argmax(dim=1)

            # top-1
            total += y.size(0)

            # top-5
            k = min(5, logits.size(1))
            topk = logits.topk(k, dim=1).indices
            top5_correct += (topk == y.unsqueeze(1)).any(dim=1).sum().item()

            all_targets.extend(y.cpu().numpy())
            all_preds.extend(pred.cpu().numpy())

    all_targets = np.array(all_targets)
    all_preds = np.array(all_preds)

    top1 = 100.0 * (all_preds == all_targets).mean()
    top5 = 100.0 * top5_correct / max(1, total)
    macro_f1 = 100.0 * f1_score(all_targets, all_preds, average="macro")
    weighted_f1 = 100.0 * f1_score(all_targets, all_preds, average="weighted")

    if num_classes is None:
        num_classes = int(max(all_targets.max(), all_preds.max())) + 1

    labels = list(range(num_classes))
    per_class_recall = recall_score(
        all_targets,
        all_preds,
        labels=labels,
        average=None,
        zero_division=0
    )

    cm = confusion_matrix(all_targets, all_preds, labels=labels)

    return {
        "top1": top1,
        "top5": top5,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "per_class_recall": per_class_recall,
        "confusion_matrix": cm,
        "targets": all_targets,
        "preds": all_preds,
    }

In [3]:
class WikiArtDataset(Dataset):
    def __init__(self, df, images_root, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.images_root = images_root
        self.transform = transform
        self.df["fullpath"] = self.df["relpath"].astype(str).apply(lambda p: os.path.join(images_root, p))

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row["fullpath"]
        label = int(row["label"])

        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label


class AttnPool1D(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.w = nn.Linear(d, 1)

    def forward(self, x):
        # x: (B, T, D)
        a = self.w(x).squeeze(-1)                # (B, T)
        a = torch.softmax(a, dim=1).unsqueeze(-1)
        return (x * a).sum(dim=1)                # (B, D)


class ResNet50_RowColBiGRU(nn.Module):
    def __init__(self, num_classes, proj_dim=320, rnn_hidden=256, dropout=0.4):
        super().__init__()

        base = resnet50(weights=ResNet50_Weights.DEFAULT)

        self.stem = nn.Sequential(
            base.conv1, base.bn1, base.relu, base.maxpool,
            base.layer1, base.layer2, base.layer3, base.layer4
        )

        self.gap = nn.AdaptiveAvgPool2d(1)

        self.proj = nn.Sequential(
            nn.Conv2d(2048, proj_dim, kernel_size=1, bias=False),
            nn.BatchNorm2d(proj_dim),
            nn.ReLU(inplace=True),
        )

        self.row_gru = nn.GRU(
            input_size=proj_dim,
            hidden_size=rnn_hidden,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.2,
        )

        self.col_gru = nn.GRU(
            input_size=proj_dim,
            hidden_size=rnn_hidden,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.2,
        )

        self.row_pool = AttnPool1D(2 * rnn_hidden)
        self.col_pool = AttnPool1D(2 * rnn_hidden)

        self.dropout = nn.Dropout(dropout)

        fusion_dim = 2048 + (2 * rnn_hidden) + (2 * rnn_hidden)
        self.head = nn.Linear(fusion_dim, num_classes)

    def forward(self, x):
        # f: (B, 2048, H, W)
        f = self.stem(x)

        # global cnn descriptor
        gap = self.gap(f).flatten(1)   # (B, 2048)

        # projected feature map
        z = self.proj(f)               # (B, C, H, W)
        B, C, H, W = z.shape

        # row-wise recurrent modeling
        # (B, C, H, W) -> (B*H, W, C)
        row_seq = z.permute(0, 2, 3, 1).contiguous().view(B * H, W, C)
        row_out, _ = self.row_gru(row_seq)                     # (B*H, W, 2h)
        row_vec = row_out.mean(dim=1).view(B, H, -1)          # (B, H, 2h)
        row_vec = self.row_pool(row_vec)                      # (B, 2h)

        # column-wise recurrent modeling
        # (B, C, H, W) -> (B*W, H, C)
        col_seq = z.permute(0, 3, 2, 1).contiguous().view(B * W, H, C)
        col_out, _ = self.col_gru(col_seq)                    # (B*W, H, 2h)
        col_vec = col_out.mean(dim=1).view(B, W, -1)          # (B, W, 2h)
        col_vec = self.col_pool(col_vec)                      # (B, 2h)

        fused = torch.cat([gap, row_vec, col_vec], dim=1)
        fused = self.dropout(fused)

        return self.head(fused)

In [6]:
import math
import numpy as np
import torch
import torch.nn as nn

def rand_bbox(size, lam):
    # size: (B, C, H, W)
    W = size[2]
    H = size[3]

    cut_rat = np.sqrt(1.0 - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)

    cx = np.random.randint(W)
    cy = np.random.randint(H)

    x1 = np.clip(cx - cut_w // 2, 0, W)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    y2 = np.clip(cy + cut_h // 2, 0, H)

    return x1, y1, x2, y2


def apply_mixup(x, y, alpha=0.3):
    lam = np.random.beta(alpha, alpha)
    index = torch.randperm(x.size(0), device=x.device)

    mixed_x = lam * x + (1.0 - lam) * x[index]
    y_a, y_b = y, y[index]

    return mixed_x, y_a, y_b, lam


def apply_cutmix(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    index = torch.randperm(x.size(0), device=x.device)

    y_a, y_b = y, y[index]

    x1, y1, x2, y2 = rand_bbox(x.size(), lam)
    x_cut = x.clone()
    x_cut[:, :, x1:x2, y1:y2] = x[index, :, x1:x2, y1:y2]

    box_area = max(0, x2 - x1) * max(0, y2 - y1)
    lam = 1.0 - box_area / float(x.size(2) * x.size(3))

    return x_cut, y_a, y_b, lam


def mixed_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1.0 - lam) * criterion(pred, y_b)


def maybe_apply_mix(x, y, p_mixup=0.35, p_cutmix=0.35, mixup_alpha=0.3, cutmix_alpha=1.0):
    r = np.random.rand()

    if r < p_mixup:
        x, y_a, y_b, lam = apply_mixup(x, y, alpha=mixup_alpha)
        return x, y_a, y_b, lam, "mixup"

    if r < p_mixup + p_cutmix:
        x, y_a, y_b, lam = apply_cutmix(x, y, alpha=cutmix_alpha)
        return x, y_a, y_b, lam, "cutmix"

    return x, y, y, 1.0, "none"

from collections import Counter
from torch.utils.data import WeightedRandomSampler

def make_weighted_sampler(labels):
    counts = Counter(labels)
    class_weights = {cls: 1.0 / count for cls, count in counts.items()}
    sample_weights = [class_weights[int(y)] for y in labels]
    sample_weights = torch.DoubleTensor(sample_weights)

    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )
    return sampler

def set_requires_grad(module, flag: bool):
    for p in module.parameters():
        p.requires_grad = flag

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

train_df = read_wikiart_csv(TRAIN_CSV)
train_df = filter_bad_images(train_df, IMAGES_ROOT)
val_df = read_wikiart_csv(VAL_CSV)

num_classes = int(max(train_df["label"].max(), val_df["label"].max())) + 1
print("num_classes:", num_classes)
print("train/val sizes:", len(train_df), len(val_df))

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(384, scale=(0.7, 1.0), ratio=(0.85, 1.15)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.10, hue=0.03),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.20, scale=(0.02, 0.12), ratio=(0.3, 3.3), value="random"),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(416),
    transforms.CenterCrop(384),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

train_dataset = WikiArtDataset(train_df, IMAGES_ROOT, transform=train_transform)
val_dataset = WikiArtDataset(val_df, IMAGES_ROOT, transform=val_transform)

train_sampler = make_weighted_sampler(train_df["label"].tolist())

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    sampler=train_sampler,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

model = ResNet50_RowColBiGRU(
    num_classes=num_classes,
    proj_dim=320,
    rnn_hidden=256,
    dropout=0.4
).to(device)

# freeze early stages, train later stages
for i in range(4):
    set_requires_grad(model.stem[i], False)
for i in range(4, 8):
    set_requires_grad(model.stem[i], True)

criterion = nn.CrossEntropyLoss(label_smoothing=0.03)

use_amp = (device.type == "cuda")
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

head_params = []
for name in ["proj", "row_gru", "col_gru", "row_pool", "col_pool", "head"]:
    head_params += list(getattr(model, name).parameters())

backbone_params = [p for p in model.stem.parameters() if p.requires_grad]
head_params = [p for p in head_params if p.requires_grad]

optimizer = optim.AdamW(
    [
        {"params": head_params, "lr": 8e-4, "weight_decay": 1e-4},
        {"params": backbone_params, "lr": 1.5e-4, "weight_decay": 1e-4},
    ]
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=15,
    eta_min=1e-6
)

epochs = 15
best_top1 = -1.0

for epoch in range(1, epochs + 1):
    model.train()
    loss_sum = 0.0
    steps = 0

    for x, y in train_loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        x_mixed, y_a, y_b, lam, mix_kind = maybe_apply_mix(
            x, y,
            p_mixup=0.35,
            p_cutmix=0.35,
            mixup_alpha=0.3,
            cutmix_alpha=1.0
        )

        with torch.amp.autocast(device_type="cuda", enabled=use_amp):
            logits = model(x_mixed)

            if mix_kind == "none":
                loss = criterion(logits, y)
            else:
                loss = mixed_criterion(criterion, logits, y_a, y_b, lam)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        loss_sum += loss.item()
        steps += 1

    scheduler.step()

    metrics = evaluate_metrics(
        model=model,
        loader=val_loader,
        device=device,
        use_amp=use_amp,
        num_classes=num_classes
    )

    print(
        f"Epoch {epoch:02d}/{epochs} | "
        f"train_loss {loss_sum/max(1, steps):.4f} | "
        f"val_top1 {metrics['top1']:.2f}% | "
        f"val_top5 {metrics['top5']:.2f}% | "
        f"macro_f1 {metrics['macro_f1']:.2f}% | "
        f"weighted_f1 {metrics['weighted_f1']:.2f}%"
    )

    if metrics["top1"] > best_top1:
        best_top1 = metrics["top1"]
        torch.save(model.state_dict(), "best_style_rowcol_bigru.pt")
        print(f"Saved best checkpoint with top1={best_top1:.2f}%")

final_metrics = evaluate_metrics(
    model=model,
    loader=val_loader,
    device=device,
    use_amp=use_amp,
    num_classes=num_classes
)

print("\n================ FINAL VALIDATION METRICS ================\n")
print(f"Top-1 Accuracy    : {final_metrics['top1']:.2f}%")
print(f"Top-5 Accuracy    : {final_metrics['top5']:.2f}%")
print(f"Macro F1 Score    : {final_metrics['macro_f1']:.2f}%")
print(f"Weighted F1 Score : {final_metrics['weighted_f1']:.2f}%")

print("\nPer-Class Recall:\n")
for i, r in enumerate(final_metrics["per_class_recall"]):
    print(f"Class {i:02d} : {100 * r:.2f}%")

from sklearn.metrics import classification_report
print("\nClassification Report:\n")
print(
    classification_report(
        final_metrics["targets"],
        final_metrics["preds"],
        digits=3
    )
)

Device: cuda
Filtered bad images: 0 removed, 57025 remaining
num_classes: 27
train/val sizes: 57025 10178
Epoch 01/15 | train_loss 1.7613 | val_top1 57.77% | val_top5 91.68% | macro_f1 58.16% | weighted_f1 57.07%
Saved best checkpoint with top1=57.77%
Epoch 02/15 | train_loss 1.3984 | val_top1 60.92% | val_top5 93.44% | macro_f1 61.48% | weighted_f1 59.94%
Saved best checkpoint with top1=60.92%
Epoch 03/15 | train_loss 1.2826 | val_top1 63.08% | val_top5 93.35% | macro_f1 63.57% | weighted_f1 62.61%
Saved best checkpoint with top1=63.08%
Epoch 04/15 | train_loss 1.1867 | val_top1 63.44% | val_top5 93.60% | macro_f1 63.56% | weighted_f1 63.03%
Saved best checkpoint with top1=63.44%
Epoch 05/15 | train_loss 1.1420 | val_top1 65.34% | val_top5 94.03% | macro_f1 66.41% | weighted_f1 65.22%
Saved best checkpoint with top1=65.34%
Epoch 06/15 | train_loss 1.0734 | val_top1 67.10% | val_top5 94.42% | macro_f1 67.82% | weighted_f1 67.22%
Saved best checkpoint with top1=67.10%
Epoch 07/15 | trai

# 